# SmartLine Colab training

Fallback for the Kaggle workflow. In Colab, set Runtime to GPU before running.
Upload or mount the labeled YOLO dataset at `/content/dataset` and clone this repository to `/content/smart-line-ai`.
The notebook stops when CUDA or labels are missing. It does not invent metrics and it does not contain credentials.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit("CUDA is not available. Select a Colab GPU runtime.")
properties = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM_GB: {properties.total_memory / (1024**3):.2f}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
import subprocess
import sys
from pathlib import Path

repo = Path("/content/smart-line-ai")
requirements = repo / "tools/training" / "requirements.txt"
if not requirements.is_file():
    raise SystemExit(
        "Clone the SmartLine repository to /content/smart-line-ai before installing pins."
    )
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
print(f"Installed pins from {requirements}")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "/content/smart-line-ai/tools/training")
import dataset_tools
import train

dataset = Path("/content/dataset")
ready, reason = train.dataset_status(dataset, "detect")
print(reason)
if not ready:
    raise SystemExit(
        "Dataset is not trainable. Place the labeled export at /content/dataset. No metric was produced."
    )
labels = dataset_tools.validate_label_files(dataset, class_count=3)
print(labels)
if labels["problems"]:
    raise SystemExit("Label validation failed. No metric was produced.")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
model.train(
    data=str(dataset / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=20,
    seed=42,
    project="/content/runs",
    name="demobox-detect",
    exist_ok=False,
)
metrics = model.val(data=str(dataset / "data.yaml"), device=0)
print(metrics.results_dict)
print(
    "Use the files written under /content/runs. Do not copy a score that was not printed by this validation."
)

In [ ]:
from pathlib import Path

best = Path("/content/runs/demobox-detect/weights/best.pt")
if not best.is_file():
    raise SystemExit(f"Trained checkpoint is missing: {best}")
from export import execute as export_checkpoint

export_checkpoint(best)
print("Download best.pt, the ONNX file, the OpenVINO directory, and results.csv.")